# Эксперимент 28: Posterior-based speaker representations

**Статья:** Automatic Detection of Speech Sound Disorder in Child Speech Using Posterior-based Speaker Representations (Автоматическая детекция речевых звуковых нарушений у детей с использованием posterior-ориентированных представлений диктора) 2022

**Ссылка:** [https://arxiv.org/abs/2203.15405](https://arxiv.org/abs/2203.15405)

**Краткое описание модели:** UBM-подход: posterior responsibilities по GMM + supervector-агрегация (Baum-Welch style statistics) -> discriminative classifier.

**Содержание статьи:** Реализован posterior-based pipeline: обучается универсальная акустическая модель, затем строятся posterior-статистики и supervector-представления utterance для детекции речевых нарушений.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
from sklearn.mixture import GaussianMixture
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, classification_report

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent))

from shared import config, data_utils
from shared.results_utils import save_result_csv

## 1. Загрузка данных и разбиение

In [ ]:
paths_train, paths_val, paths_test, y_train, y_val, y_test, letters_train, letters_val, letters_test = data_utils.get_splits()
print(f"Train: {len(paths_train)}, Val: {len(paths_val)}, Test: {len(paths_test)}")
n_letters = letters_train.shape[1]

Train: 1942, Val: 417, Test: 417


## 2. Подготовка признаков / представлений

In [ ]:
def extract_seq(path, n_mfcc=20, max_frames=320):
    y, sr = data_utils.load_audio(path, sr=config.TARGET_SR, max_sec=config.MAX_DURATION_SEC)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, hop_length=config.HOP_LENGTH, n_fft=config.WIN_LENGTH).T
    if mfcc.shape[0] > max_frames:
        idx = np.linspace(0, mfcc.shape[0] - 1, max_frames).astype(int)
        mfcc = mfcc[idx]
    return mfcc.astype(np.float32)

seq_train = [extract_seq(p) for p in paths_train]
seq_val   = [extract_seq(p) for p in paths_val]
seq_test  = [extract_seq(p) for p in paths_test]

## 3. Обучение, оценка и сохранение результата

In [ ]:
ubm = GaussianMixture(n_components=16, covariance_type="diag", reg_covar=1e-4, random_state=config.RANDOM_STATE, max_iter=200)
ubm.fit(np.vstack(seq_train))

def posterior_supervector(seq):
    post = ubm.predict_proba(seq)
    hist = post.mean(axis=0)
    ent = -np.sum(post * np.log(post + 1e-8), axis=1)
    ent_stats = np.array([ent.mean(), ent.std(), ent.max()], dtype=np.float32)
    w = post.sum(axis=0)[:, None] + 1e-6
    first = (post.T @ seq) / w
    supervec = first.reshape(-1).astype(np.float32)
    return np.concatenate([hist.astype(np.float32), ent_stats, supervec], axis=0)

X_train = np.stack([posterior_supervector(s) for s in seq_train])
X_val   = np.stack([posterior_supervector(s) for s in seq_val])
X_test  = np.stack([posterior_supervector(s) for s in seq_test])
if letters_train.shape[1] > 0:
    X_train = np.hstack([X_train, letters_train])
    X_val   = np.hstack([X_val, letters_val])
    X_test  = np.hstack([X_test, letters_test])

pipe = Pipeline([
    ("scaler", StandardScaler()), 
    ("clf", LogisticRegression(class_weight="balanced", max_iter=4000, solver="liblinear"))
])
param_grid = {"clf__C": [0.1, 0.3, 1.0, 3.0, 10.0]}
grid = GridSearchCV(pipe, param_grid, cv=5, scoring="f1_macro", refit=True, n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)
clf = grid.best_estimator_

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]
accuracy = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_bad = f1_score(y_test, y_pred, average="binary", pos_label=config.CLASS_BAD)
roc_auc = roc_auc_score(y_test, y_proba)
precision_bad = precision_score(y_test, y_pred, zero_division=0, pos_label=config.CLASS_BAD)
recall_bad = recall_score(y_test, y_pred, zero_division=0, pos_label=config.CLASS_BAD)

print(classification_report(y_test, y_pred, target_names=config.CLASS_NAMES))
display(pd.DataFrame([{"accuracy": accuracy, "f1_macro": f1_macro, "f1_bad": f1_bad, "roc_auc": roc_auc, "precision_bad": precision_bad, "recall_bad": recall_bad}]))

save_result_csv(
    exp_dir=exp_dir, 
    experiment_id="exp_28_posterior_speaker_repr", 
    experiment_name="Posterior-based speaker representations", 
    model="UBM posterior supervector + LogReg", 
    accuracy=accuracy, 
    f1_macro=f1_macro, 
    f1_bad=f1_bad, 
    roc_auc=roc_auc, 
    precision_bad=precision_bad, 
    recall_bad=recall_bad, 
    notes=f"ubm_k=16,posterior_supervector,grid={grid.best_params_}", 
    num_params=None, 
    train_time_sec=None
)

Fitting 5 folds for each of 5 candidates, totalling 25 fits
              precision    recall  f1-score   support

        good       0.83      0.75      0.79       282
         bad       0.56      0.68      0.62       135

    accuracy                           0.73       417
   macro avg       0.70      0.71      0.70       417
weighted avg       0.74      0.73      0.73       417



,accuracy,f1_macro,f1_bad,roc_auc,precision_bad,recall_bad
0,0.726619,0.702382,0.61745,0.777357,0.564417,0.681481


Результат сохранён в result.csv текущего эксперимента
